# HW2 - Final pipeline

Uses the results of `hw2.ipynb` (EDA, Faster R-CNN, RetinaNet, YOLOv8, RT-DETR).

In [ ]:
SLUG = "multiple-object-video-detection-hw-2-ucu-ai-2026"
COMP = "/content/comp"
SRC = "/content/drive/MyDrive/hw2_own"
WORK = "/content/drive/MyDrive/hw2_final"
DATA = "/content/data"

SEED = 42
TRAIN_STEP = 20
VAL_STEP = 5
MAX_SIDE = 1600
IMGSZ = 1280

DEBUG = False

EPOCHS_YOLO = 1 if DEBUG else 12
EPOCHS_TV = 1 if DEBUG else 4
EPOCHS_DETR = 1 if DEBUG else 6

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi -L
!pip install -q -U ultralytics pycocotools kaggle

In [ ]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
if not os.path.exists(f"{WORK}/results.csv"):
    os.makedirs(f"{WORK}/runs", exist_ok=True)
    shutil.copy(f"{SRC}/results.csv", WORK)
    for name in ["yolo_base", "yolo_1600", "yolo_p2", "final_mask", "final_yolov8m"]:
        if os.path.exists(f"{SRC}/runs/{name}/weights/best.pt"):
            shutil.copytree(f"{SRC}/runs/{name}", f"{WORK}/runs/{name}", dirs_exist_ok=True)
    shutil.copytree(f"{SRC}/subs", f"{WORK}/subs", dirs_exist_ok=True)
print(os.listdir(f"{WORK}/runs"))

In [ ]:
import os, glob
from getpass import getpass

if not glob.glob(f"{COMP}/**/test_images.csv", recursive=True):
    os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API token: ").strip()
    !kaggle competitions download -c {SLUG} -p /content/raw
    !unzip -q -o /content/raw/{SLUG}.zip -d {COMP}
    !rm -f /content/raw/{SLUG}.zip

found = glob.glob(f"{COMP}/**/test_images.csv", recursive=True)
assert found, "Download failed: check the token and that you joined the competition (see the kaggle message above)"
COMP = os.path.dirname(found[0])
print(COMP, os.listdir(COMP))

In [ ]:
import os, io, json, time, math, random, contextlib, logging, shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image, ImageReadMode
from torchvision.ops import box_iou, batched_nms, generalized_box_iou_loss
from ultralytics import YOLO, RTDETR

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_all()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
for d in ["runs", "figs", "subs"]:
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Data

In [ ]:
def load_coco(split):
    with open(f"{COMP}/{split}/annotations_coco.json") as f:
        coco = json.load(f)
    imgs = pd.DataFrame(coco["images"])
    anns = pd.DataFrame(coco["annotations"])
    anns[["x", "y", "w", "h"]] = pd.DataFrame(anns["bbox"].tolist(), index=anns.index)
    anns = anns.merge(imgs[["id", "video", "width", "height", "frame_id"]].rename(columns={"id": "image_id"}),
                      on="image_id")
    imgs["split"] = split
    anns["split"] = split
    return coco, imgs, anns

coco_tr, imgs_tr, anns_tr = load_coco("train")
coco_va, imgs_va, anns_va = load_coco("val")
imgs = pd.concat([imgs_tr, imgs_va], ignore_index=True)
anns = pd.concat([anns_tr, anns_va], ignore_index=True)
anns["category"] = anns.category_id.map({c["id"]: c["name"] for c in coco_tr["categories"]})
test_df = pd.read_csv(f"{COMP}/test_images.csv")

print("train:", len(imgs_tr), "frames,", imgs_tr.video.nunique(), "videos")
print("val:  ", len(imgs_va), "frames,", imgs_va.video.nunique(), "videos")
print("test: ", len(test_df), "frames,", test_df["clip"].nunique(), "clips")
print("videos both in train and val:", set(imgs_tr.video) & set(imgs_va.video))

In [ ]:
def prepare(out, mask_ignore=False):
    if os.path.exists(f"{out}/data.yaml"):
        return out
    for split, step, coco in [("train", TRAIN_STEP, coco_tr), ("val", VAL_STEP, coco_va)]:
        images = [dict(im) for im in coco["images"] if (im["frame_id"] - 1) % step == 0]
        keep = {im["id"] for im in images}
        ann = [a for a in coco["annotations"] if a["image_id"] in keep]
        ign = defaultdict(list)
        for a in ann:
            if a["iscrowd"] == 1 and a["category_id"] == 1:
                ign[a["image_id"]].append(a["bbox"])

        def one(im):
            s = min(1.0, MAX_SIDE / max(im["width"], im["height"]))
            im["scale"] = s
            img = Image.open(f"{COMP}/{split}/{im['file_name']}").convert("RGB")
            if s < 1:
                img = img.resize((round(im["width"] * s), round(im["height"] * s)), Image.BILINEAR)
            if mask_ignore and split == "train":
                d = ImageDraw.Draw(img)
                for x, y, w, h in ign[im["id"]]:
                    d.rectangle([x * s, y * s, (x + w) * s, (y + h) * s], fill=(114, 114, 114))
            dst = f"{out}/{split}/{im['file_name']}"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            img.save(dst, quality=95)
            lab_src = f"{COMP}/{split}/{im['file_name']}".replace("/images/", "/labels/").replace(".jpg", ".txt")
            lab_dst = dst.replace("/images/", "/labels/").replace(".jpg", ".txt")
            os.makedirs(os.path.dirname(lab_dst), exist_ok=True)
            if os.path.exists(lab_src):
                shutil.copy(lab_src, lab_dst)
            else:
                open(lab_dst, "w").close()

        with ThreadPoolExecutor(8) as ex:
            list(ex.map(one, images))
        with open(f"{out}/{split}_gt.json", "w") as f:
            json.dump({"images": images, "annotations": ann, "categories": coco["categories"]}, f)
        print(split, len(images), "frames")
    with open(f"{out}/data.yaml", "w") as f:
        f.write(f"path: {out}\ntrain: train/images\nval: val/images\nnames: {CLASSES}\n")
    return out

t = time.time()
D0 = prepare(f"{DATA}/base")
GT = f"{D0}/val_gt.json"
print(f"{time.time() - t:.0f}s")

In [ ]:
def val_items(d):
    gt = json.load(open(f"{d}/val_gt.json"))
    return [(im["id"], f"{d}/val/{im['file_name']}", im["scale"]) for im in gt["images"]]

VAL = val_items(D0)
TEST = [(r.image_id, f"{COMP}/{r.file_name}", 1.0) for r in test_df.itertuples()]
if DEBUG:
    VAL, TEST = VAL[:30], TEST[:30]

_gt = json.load(open(GT))
GT_OBJ, GT_IGN = defaultdict(list), defaultdict(list)
for a in _gt["annotations"]:
    if a["iscrowd"] == 0:
        GT_OBJ[a["image_id"]].append(a["bbox"] + [a["category_id"]])
    elif a["category_id"] == 1:
        GT_IGN[a["image_id"]].append(a["bbox"])
VAL_IMG = {im["id"]: im for im in _gt["images"]}
print(len(VAL), "val frames,", len(TEST), "test frames")

## Evaluation and helpers

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate(preds, gt_json=GT):
    if len(preds) == 0:
        return dict(mAP=0, AP50=0, AP75=0, AP_small=0, AP_medium=0, AP_large=0)
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json)
        E = COCOeval(gt, gt.loadRes(preds), "bbox")
        E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    r = dict(mAP=s[0], AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4], AP_large=s[5])
    prec = E.eval["precision"]
    for k, c in enumerate(E.params.catIds):
        p = prec[:, :, k, 0, -1]
        r["AP_" + CLASSES[c - 1]] = p[p > -1].mean() if (p > -1).any() else float("nan")
    return {k: round(float(v), 4) for k, v in r.items()}

RESULTS = pd.read_csv(f"{WORK}/results.csv", index_col=0).to_dict("index") if os.path.exists(f"{WORK}/results.csv") else {}
def log(name, family, metrics, **extra):
    RESULTS[name] = dict(family=family, **metrics, **extra)
    pd.DataFrame(RESULTS).T.to_csv(f"{WORK}/results.csv")
    return pd.DataFrame([RESULTS[name]], index=[name])

def speed(predict_one, paths, n=50):
    for p in paths[:5]:
        predict_one(p)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t = time.time()
    for p in paths[5:5 + n]:
        predict_one(p)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.time() - t) / len(paths[5:5 + n]) * 1000
    return dict(ms=round(ms, 1), FPS=round(1000 / ms, 1))

def write_sub(preds, path):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    rows = []
    for iid in test_df.image_id:
        ps = sorted(by[iid], key=lambda p: -p["score"])[:500]
        rows.append(" ".join(f"{p['category_id'] - 1} {p['score']:.4f} {p['bbox'][0]:.1f} {p['bbox'][1]:.1f} "
                             f"{p['bbox'][2]:.1f} {p['bbox'][3]:.1f}" for p in ps))
    pd.DataFrame({"image_id": test_df.image_id, "PredictionString": rows}).to_csv(path, index=False)
    print("saved", path)

COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms", "FPS"]

In [ ]:
def iou_xywh(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = torch.tensor(np.c_[a[:, :2], a[:, :2] + a[:, 2:]])
    b = torch.tensor(np.c_[b[:, :2], b[:, :2] + b[:, 2:]])
    return box_iou(a, b).numpy()

def match_frame(iid, preds, thr=0.3):
    g = np.array(GT_OBJ[iid], float).reshape(-1, 5)
    ign = np.array(GT_IGN[iid], float).reshape(-1, 4)
    d = sorted([p for p in preds if p["score"] >= thr], key=lambda p: -p["score"])
    found = np.zeros(len(g), bool)
    types = []
    for p in d:
        iou = iou_xywh([p["bbox"]], g[:, :4])[0] if len(g) else np.zeros(0)
        same = g[:, 4] == p["category_id"] if len(g) else np.zeros(0, bool)
        cand = np.where(same & ~found & (iou >= 0.5))[0]
        if len(cand):
            found[cand[iou[cand].argmax()]] = True
            types.append("TP")
        elif len(ign) and iou_xywh([p["bbox"]], ign).max() > 0:
            x, y, w, h = p["bbox"]
            cx, cy = x + w / 2, y + h / 2
            inside = ((ign[:, 0] <= cx) & (cx <= ign[:, 0] + ign[:, 2]) & (ign[:, 1] <= cy) & (cy <= ign[:, 1] + ign[:, 3])).any()
            types.append("ignored" if inside else "background")
        elif len(g) and (iou[same] >= 0.5).any():
            types.append("duplicate")
        elif len(g) and (iou >= 0.5).any():
            types.append("wrong class")
        elif len(g) and (iou[same] >= 0.1).any():
            types.append("bad box")
        else:
            types.append("background")
    return d, types, g, found

def error_analysis(preds, thr=0.3):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    types, miss = defaultdict(int), defaultdict(lambda: [0, 0])
    worst = []
    for iid in VAL_IMG:
        d, t, g, found = match_frame(iid, by[iid], thr)
        for x in t:
            types[x] += 1
        for (x, y, w, h, c), f in zip(g, found):
            s = "small" if w * h < 32**2 else "medium" if w * h < 96**2 else "large"
            miss[s][0 if f else 1] += 1
        worst.append(((~found).sum() + sum(x not in ("TP", "ignored") for x in t), iid))
    print("detections:", dict(types))
    print("GT found / missed:", {k: f"{v[0]}/{v[1]} ({v[1] / max(sum(v), 1):.0%} missed)" for k, v in miss.items()})
    return [iid for _, iid in sorted(worst, reverse=True)]

def show_errors(preds, ids, title, thr=0.3):
    by = defaultdict(list)
    for p in preds:
        by[p["image_id"]].append(p)
    fig, axes = plt.subplots(len(ids) // 2, 2, figsize=(20, 6 * (len(ids) // 2)))
    for ax, iid in zip(np.ravel(axes), ids):
        im = VAL_IMG[iid]; s = im["scale"]
        ax.imshow(Image.open(f"{D0}/val/{im['file_name']}"))
        d, t, g, found = match_frame(iid, by[iid], thr)
        for x, y, w, h in GT_IGN[iid]:
            ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, color="grey", alpha=0.4))
        for (x, y, w, h, c), f in zip(g, found):
            if not f:
                ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, fill=False, ec="yellow", lw=1.2))
        for p, tt in zip(d, t):
            if tt != "ignored":
                x, y, w, h = p["bbox"]
                ax.add_patch(patches.Rectangle((x * s, y * s), w * s, h * s, fill=False, lw=0.8,
                                               ec="lime" if tt == "TP" else "red"))
        ax.set_title(f"{im['video']} #{im['frame_id']}  TP={t.count('TP')} FP={sum(x not in ('TP', 'ignored') for x in t)} "
                     f"missed={(~found).sum()}", fontsize=10)
        ax.axis("off")
    fig.suptitle(title + "  (green TP, red FP, yellow missed, grey ignore)")
    plt.tight_layout(); plt.savefig(f"{WORK}/figs/{title}.png", dpi=80); plt.show()

def size_plot(df, title):
    df[["AP_small", "AP_medium", "AP_large", "mAP"]].astype(float).plot.bar(figsize=(9, 4), rot=0, title=title)
    plt.tight_layout(); plt.savefig(f"{WORK}/figs/{title}.png", dpi=110); plt.show()

In [ ]:
def ul_train(name, src, imgsz, epochs, batch, load=None, cls=YOLO, data=None, **kw):
    run = f"{WORK}/runs/{name}"
    best, csv = f"{run}/weights/best.pt", f"{run}/results.csv"
    if os.path.exists(best) and os.path.exists(csv) and len(pd.read_csv(csv)) >= epochs:
        print(name, "already trained - loading")
        return cls(best)
    model = cls(src)
    if load:
        model.load(load)
    logging.getLogger("ultralytics").setLevel(logging.WARNING)
    t = time.time()
    model.train(data=data or f"{D0}/data.yaml", imgsz=imgsz, epochs=epochs, batch=batch, seed=SEED,
                project="/content/runs", name=name, exist_ok=True, workers=2, plots=False,
                fraction=0.1 if DEBUG else 1.0, **kw)
    print(f"{name}: {(time.time() - t) / 60:.0f} min")
    tr = model.trainer
    ckpt = str(tr.best if tr.best.exists() else tr.last)
    shutil.copytree(tr.save_dir, run, dirs_exist_ok=True)
    shutil.copy(ckpt, best)
    print("saved:", best, os.path.getsize(best) // 2**20, "MB")
    return cls(ckpt)

def ul_predict(model, items, imgsz, iou=0.6, augment=False, bs=8):
    preds = []
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        res = model.predict([p for _, p, _ in chunk], imgsz=imgsz, conf=0.001, iou=iou, max_det=500,
                            augment=augment, verbose=False)
        for (key, _, s), r in zip(chunk, res):
            for (x1, y1, x2, y2), sc, c in zip((r.boxes.xyxy.cpu().numpy() / s).tolist(),
                                               r.boxes.conf.tolist(), r.boxes.cls.tolist()):
                preds.append(dict(image_id=key, category_id=int(c) + 1, score=round(sc, 5),
                                  bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)]))
    return preds

def ul_speed(model, imgsz):
    return speed(lambda p: model.predict(p, imgsz=imgsz, conf=0.001, max_det=500, verbose=False),
                 [p for _, p, _ in VAL])

def ul_curve(name):
    df = pd.read_csv(f"{WORK}/runs/{name}/results.csv")
    df.columns = [c.strip() for c in df.columns]
    return df

In [ ]:
YOLO_RUNS = {
    "yolo_base": dict(src="yolov8s.pt", imgsz=1280, batch=8),
    "yolo_1600": dict(src="yolov8s.pt", imgsz=1600, batch=4),
    "yolo_p2":   dict(src="yolov8s-p2.yaml", load="yolov8s.pt", imgsz=1280, batch=4),
}
yolo_preds = {}

def run_yolo(name):
    c = YOLO_RUNS[name]
    m = ul_train(name, c["src"], c["imgsz"], EPOCHS_YOLO, c["batch"], load=c.get("load"),
                 optimizer="SGD", lr0=0.01, close_mosaic=3, max_det=500, cache=False)
    yolo_preds[name] = ul_predict(m, VAL, c["imgsz"])
    return log(name, "yolo", {**evaluate(yolo_preds[name]), **ul_speed(m, c["imgsz"])}, imgsz=c["imgsz"], src=c["src"])

In [ ]:
yo = pd.DataFrame({k: v for k, v in RESULTS.items() if v["family"] == "yolo"}).T
display(pd.DataFrame(RESULTS).T[["family"] + COLS])

## 10. Final pipeline

In [ ]:
start = yo.mAP.astype(float).idxmax()
SC = YOLO_RUNS[start]
model = YOLO(f"{WORK}/runs/{start}/weights/best.pt")
print("start from", start)

VAL_LOW = []
for key, p, s in VAL:
    im = VAL_IMG[key]
    f = 680 / max(im["width"], im["height"])
    dst = p.replace("/val/", "/val680/")
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        Image.open(p).resize((round(im["width"] * f), round(im["height"] * f)), Image.BILINEAR).save(dst)
    VAL_LOW.append((key, dst, f))

def tiles_predict(m, items, imgsz, iou=0.6):
    preds = []
    for key, p, s in items:
        img = np.array(Image.open(p).convert("RGB"))[:, :, ::-1]
        H, W = img.shape[:2]
        tw, th = int(W / 1.8), int(H / 1.8)
        crops = [(img[y:y + th, x:x + tw], x, y) for y in [0, H - th] for x in [0, W - tw]] + [(img, 0, 0)]
        B, S_, C = [], [], []
        for (crop, x0, y0), r in zip(crops, m.predict([np.ascontiguousarray(c[0]) for c in crops], imgsz=imgsz,
                                                      conf=0.001, iou=iou, max_det=500, verbose=False)):
            B.append(r.boxes.xyxy.cpu() + torch.tensor([x0, y0, x0, y0]))
            S_.append(r.boxes.conf.cpu()); C.append(r.boxes.cls.cpu())
        B, S_, C = torch.cat(B), torch.cat(S_), torch.cat(C)
        keep = batched_nms(B, S_, C.long(), iou)[:500]
        for (x1, y1, x2, y2), sc, c in zip((B[keep] / s).tolist(), S_[keep].tolist(), C[keep].tolist()):
            preds.append(dict(image_id=key, category_id=int(c) + 1, score=round(sc, 5),
                              bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)]))
    return preds

FINAL = {}
def final_eval(name, fn):
    m = evaluate(fn(VAL))
    m["mAP_680"] = evaluate(fn(VAL_LOW))["mAP"]
    m["score"] = round(0.84 * m["mAP"] + 0.16 * m["mAP_680"], 4)
    FINAL[name] = m
    return pd.DataFrame([m], index=[name])[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

### Inference strategies (same weights)

In [ ]:
INFER = {
    "plain":    lambda m, it: ul_predict(m, it, SC["imgsz"]),
    "nms0.5":   lambda m, it: ul_predict(m, it, SC["imgsz"], iou=0.5),
    "nms0.7":   lambda m, it: ul_predict(m, it, SC["imgsz"], iou=0.7),
    "size+320": lambda m, it: ul_predict(m, it, SC["imgsz"] + 320),
    "tta":      lambda m, it: ul_predict(m, it, SC["imgsz"], augment=True),
    "tiles":    lambda m, it: tiles_predict(m, it, SC["imgsz"]),
}
for k, fn in INFER.items():
    final_eval("A_" + k, lambda it, fn=fn: fn(model, it))
inf = pd.DataFrame(FINAL).T
best_inf = inf.score.astype(float).idxmax()[2:]
print("best inference:", best_inf)
inf

### Data strategy: grey out ignored regions in train frames

In [ ]:
D_MASK = prepare(f"{DATA}/mask", mask_ignore=True)
m_mask = ul_train("final_mask", SC["src"], SC["imgsz"], EPOCHS_YOLO, SC["batch"], load=SC.get("load"),
                  data=f"{D_MASK}/data.yaml", optimizer="SGD", lr0=0.01, close_mosaic=3, max_det=500, cache=False)
final_eval("B_mask", lambda it: ul_predict(m_mask, it, SC["imgsz"]))

### Model change: YOLOv8m

In [ ]:
use_mask = FINAL["B_mask"]["score"] > FINAL["A_plain"]["score"]
src_m = SC["src"].replace("yolov8s", "yolov8m")
m_big = ul_train("final_yolov8m", src_m, SC["imgsz"], EPOCHS_YOLO, SC["batch"],
                 load="yolov8m.pt" if SC.get("load") else None,
                 data=f"{D_MASK if use_mask else D0}/data.yaml", optimizer="SGD", lr0=0.01, close_mosaic=3,
                 max_det=500, cache=False)
final_eval("C_yolov8m", lambda it: ul_predict(m_big, it, SC["imgsz"]))
pd.DataFrame(FINAL).T[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

### Final submission

In [ ]:
cands = {"A_plain": model, "B_mask": m_mask, "C_yolov8m": m_big}
final_name = max(cands, key=lambda k: FINAL[k]["score"])
print("final:", final_name, "+", best_inf)
final_eval("final", lambda it: INFER[best_inf](cands[final_name], it))
write_sub(INFER[best_inf](cands[final_name], TEST), f"{WORK}/submission.csv")
pd.DataFrame(FINAL).T[["score", "mAP", "mAP_680", "AP50", "AP_small"]]

## 11. Summary

In [ ]:
allm = pd.DataFrame(RESULTS).T
display(allm[["family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms", "FPS"]])
best_fam = allm.loc[allm.groupby("family").mAP.apply(lambda s: s.astype(float).idxmax())]
size_plot(best_fam.set_index("family"), "best_per_family_by_size")
pd.DataFrame(FINAL).T.to_csv(f"{WORK}/final_results.csv")
print(os.listdir(f"{WORK}/subs"))